# GIST -> SAM-Med3D (Steps 1–3) via med3pipe

This notebook reproduces the first 3 steps of `gist_tabpfn_end_to_end_corrected.ipynb` by importing
and calling the reusable functions in `med3pipe.prepare` (no CLI).

Steps:
1. Discover raw GIST cases in `gist/`.
2. Prepare SAM-Med3D-ready folders: `imagesTr/labelsTr` with merged lesions, aligned geometry,
   and binary labels.
3. Create a validation split by copying subset into `imagesVal/labelsVal` (seed 2025, split 0.8/0.2).


In [20]:
# Paths and imports
from pathlib import Path
import sys

def find_project_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / 'gist').is_dir() and (cand / 'SAM-Med3D-main').is_dir():
            return cand
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
# Make the project importable (so we can `import med3pipe`)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ROOT = PROJECT_ROOT / 'gist'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)


PROJECT_ROOT: C:\Users\cahel\Desktop\Med3Tab-PFN
DATASET_ROOT: C:\Users\cahel\Desktop\Med3Tab-PFN\gist


## 1–2) Prepare dataset into SAM-Med3D format
- Merge multiple lesions per case (images: max, labels: union)
- Align image to label geometry
- Binarize labels (>0)
- Write to `data/train/<category>/<ct_name>/{imagesTr,labelsTr}`


In [21]:
from med3pipe.data import prepare_for_sam3d, find_default_sam3d_root

SAM3D_ROOT = find_default_sam3d_root(PROJECT_ROOT)
category = 'gist'
ct_name = 'ct_GIST'

n_prepared, paths = prepare_for_sam3d(
    dataset_root=DATASET_ROOT,
    sam3d_root=SAM3D_ROOT,
    category=category,
    ct_name=ct_name,
    case_glob=None,  # default GIST pattern will be used
)
print('Prepared cases:', n_prepared)
print('Train images folder:', paths.images_tr)
print('Train labels folder:', paths.labels_tr)


Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Prepared cases: 246
Train images folder: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Train labels folder: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\labelsTr


## 3) Create validation split
- Copy a subset to `imagesVal/labelsVal` (non-destructive)
- Seed 2025, split 0.8 train / 0.2 val (to match the original)


In [22]:
from med3pipe.data import split_validation

n_train, n_val = split_validation(paths, split_ratio=0.8, seed=2025, copy=True)
print('Train:', n_train, '| Val:', n_val)
print('Val images folder:', paths.images_val)
print('Val labels folder:', paths.labels_val)


Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
Train: 196 | Val: 50
Val images folder: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Val labels folder: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\labelsVal


In [23]:
# Quick checks
train_imgs = sorted(paths.images_tr.glob('*.nii.gz'))
train_lbls = sorted(paths.labels_tr.glob('*.nii.gz'))
val_imgs = sorted(paths.images_val.glob('*.nii.gz'))
val_lbls = sorted(paths.labels_val.glob('*.nii.gz'))

print('Train imgs/labels:', len(train_imgs), len(train_lbls))
print('Val imgs/labels  :', len(val_imgs), len(val_lbls))
print('Sample train files:', [p.name for p in train_imgs[:5]])
print('Sample val files  :', [p.name for p in val_imgs[:5]])


Train imgs/labels: 246 246
Val imgs/labels  : 78 78
Sample train files: ['GIST-001_CT.nii.gz', 'GIST-002_CT.nii.gz', 'GIST-003_CT.nii.gz', 'GIST-004_CT.nii.gz', 'GIST-005_CT.nii.gz']
Sample val files  : ['GIST-001_CT.nii.gz', 'GIST-007_CT.nii.gz', 'GIST-008_CT.nii.gz', 'GIST-012_CT.nii.gz', 'GIST-013_CT.nii.gz']


# Build model (optionally load checkpoint)

In [24]:
from pathlib import Path
import torch
from med3pipe import (
    find_default_sam3d_root, build_sam3d_model,
    default_feature_dirs, extract_embeddings_train_val,
    load_roi_features, load_labels_from_sheet, build_y,
)

SAM3D_ROOT = find_default_sam3d_root()
ckpt = SAM3D_ROOT / "ckpt" / "sam_med3d_turbo.pth"  # optional
model = build_sam3d_model(sam3d_root=SAM3D_ROOT, checkpoint=ckpt, model_type="vit_b_ori")

# Fine-Tuning

In [25]:
from med3pipe import finetune_sam3d
proc = finetune_sam3d(paths, model_type="vit_b_ori", checkpoint=ckpt, device="cuda", num_epochs=2, batch_size=4)
proc.wait()

1

# Extract embeddings

In [26]:

# Extract embeddings for TRAIN and VAL
feat_dirs = default_feature_dirs(SAM3D_ROOT, category=paths.category, ct_name=paths.ct_name)
extract_embeddings_train_val(paths, model, sam3d_root=SAM3D_ROOT)

To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Extracted 25 embeddings ...
Extracted 50 embeddings ...
Extracted 75 embeddings ...
Extracted 100 embeddings ...
Extracted 125 embeddings ...
Extracted 150 embeddings ...
Extracted 175 embeddings ...
Extracted 200 embeddings ...
Extracted 225 embeddings ...
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 78 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Extracted 25 embeddings ...
Extracted 50 embeddings ...
Extracted 75 embeddings ...
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST


FeatureDirs(train_dir=WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/features/gist/ct_GIST_train'), val_dir=WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN/SAM-Med3D-main/SAM-Med3D-main/features/gist/ct_GIST'))

# ROI-Pooling with SAM-Med3D embeddings

In [27]:
# ROI-pool to per-case vectors
X_train, ids_train = load_roi_features(feat_dirs.train_dir, paths.labels_tr)
X_val,   ids_val   = load_roi_features(feat_dirs.val_dir,   paths.labels_val)

# Load Labels from csv

In [28]:
df, lab_map = load_labels_from_sheet(Path(r"C:\Users\cahel\Desktop\Med3Tab-PFN\gist\sheet.csv"))  # dataset_name='GIST' by default

# ids_train / ids_val must already be defined from ROI features
y_train, missing_tr = build_y(ids_train, lab_map)
y_val,   missing_va = build_y(ids_val,   lab_map)

print("y_train:", y_train.shape, "y_val:", y_val.shape)
if missing_tr or missing_va:
    print("WARNING missing labels. train:", len(missing_tr), "val:", len(missing_va))

y_train: (246,) y_val: (78,)


# TabPFN

In [29]:
from med3pipe import tabpfn_pipeline

# Option A: One-liner pipeline with default saving
res = tabpfn_pipeline(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    ids_val=ids_val,
    category=paths.category,  # e.g., "gist"
    ct_name=paths.ct_name,    # e.g., "ct_GIST"
)

print("Saved outputs under:", res["out_dir"])
print("Preproc dir:", res["preproc_dir"])
print("Predictions:", res["pred_path"])
print("Metrics:", res["metrics_path"])
print("Report:", res["report_path"])

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tabpfn\classifier.py:465: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(


Saved outputs under: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_152219
Preproc dir: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_152219\preproc
Predictions: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_152219\tabpfn_val_predictions.csv
Metrics: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_152219\tabpfn_metrics.json
Report: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_152219\tabpfn_classification_report.txt


In [30]:
print("\nTest (VAL) metrics:")
print(res["metrics"])


Test (VAL) metrics:
{'accuracy': 0.9871794871794872, 'macro_f1': 0.9871265885459646, 'roc_auc': 0.9993408042188531, 'confusion_matrix': [[36, 1], [0, 41]]}


# Everything at once

In [31]:
from pathlib import Path
from med3pipe import run_end_to_end

res = run_end_to_end(
    dataset_root=Path(r"C:\Users\cahel\Desktop\Med3Tab-PFN\gist"),
    category="gist",
    ct_name="ct_GIST",
    sheet_csv=Path(r"C:\Users\cahel\Desktop\Med3Tab-PFN\gist\sheet.csv"),  # force exact file
)

print("TabPFN metrics:", res.tabpfn["metrics"])
print("Saved run dir:", res.tabpfn["out_dir"])
print("Predictions CSV:", res.tabpfn["pred_path"])
print("Report TXT:", res.tabpfn["report_path"])

Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST_train
To extract: 78 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST\imagesVal
Done extracting to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\features\gist\ct_GIST


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tabpfn\classifier.py:465: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(


TabPFN metrics: {'accuracy': 0.9871794871794872, 'macro_f1': 0.9871265885459646, 'roc_auc': 0.9993408042188531, 'confusion_matrix': [[36, 1], [0, 41]]}
Saved run dir: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_153435
Predictions CSV: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_153435\tabpfn_val_predictions.csv
Report TXT: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\tabpfn_runs\gist_ct_GIST_20250916_153435\tabpfn_classification_report.txt


# Swin transformer

Volumetric CT/MRI (medical imaging)

Swin UNETR (MONAI) — a 3D Swin-Transformer encoder + U-Net-style decoder, widely used for CT/MRI segmentation; official code, docs, tutorials, and pretrained weights are available. 

Paper: Self-Supervised Pre-Training of Swin Transformers for 3D Medical Image Analysis (CVPR 2022), which introduced Swin UNETR and released code. 


Related: “Swin-Unet3D” variants (3D Swin blocks in encoder/decoder). 


Why these are sensible for STS

Published STS MRI classification has successfully used CNN backbones (e.g., DenseNet-161 for grade G1 vs G2/G3, external validation). Your 3D DenseNet/ResNet baselines match that recipe but operate on volumes.

# Other models

In [32]:
import sys, site, os

# 1) Ignore user site for this session
os.environ["PYTHONNOUSERSITE"] = "1"  # takes effect on restart; we also patch sys.path below
user_site = site.getusersitepackages()
sys.path = [p for p in sys.path if p != user_site]

# 2) Verify imports now come from your env
import numpy as np, pandas as pd
print("numpy:", np.__version__)
print("numpy file:", np.__file__)
print("pandas:", pd.__version__)
print("pandas file:", pd.__file__)

numpy: 2.2.6
numpy file: c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\numpy\__init__.py
pandas: 2.3.2
pandas file: c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\pandas\__init__.py


In [34]:
from importlib import reload
import med3pipe
from importlib import reload, import_module
import med3pipe

# Pick up the new __init__ change that exposes `vision`
reload(med3pipe)

# Import the submodule to ensure med3pipe.vision.v3d is loaded
v3d = import_module('med3pipe.vision.v3d')

# Now reload the submodule if you’ve edited it
reload(v3d)

# Optionally reload the top-level package to refresh re-exports
reload(med3pipe)   # reload submodule first
reload(med3pipe)            # then reload the package, which re-exports the updated symbols
from importlib import reload
import med3pipe.vision.v3d as v3d, med3pipe
reload(v3d); reload(med3pipe)
from med3pipe import Sam3DPaths, Train3DConfig, train_eval_densenet121_3d

In [35]:
import monai, torch
print("MONAI version:", monai.__version__)

from monai.networks.nets import DenseNet121 as MDN
print("DenseNet121 import OK:", MDN)

MONAI version: 1.5.0
DenseNet121 import OK: <class 'monai.networks.nets.densenet.DenseNet121'>


In [36]:
from pathlib import Path
from med3pipe import Sam3DPaths, find_default_sam3d_root

SAM3D_ROOT = find_default_sam3d_root()  # auto-detects <PROJECT_ROOT>/SAM-Med3D-main/SAM-Med3D-main
paths = Sam3DPaths(sam3d_root=SAM3D_ROOT, category="gist", ct_name="ct_GIST")

print("imagesTr:", paths.images_tr.resolve(), paths.images_tr.exists())
sorted_ids = [p.stem for p in sorted(paths.images_tr.glob("*.nii.gz"))[:10]]
sorted_ids

imagesTr: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST\imagesTr True


['GIST-001_CT.nii',
 'GIST-002_CT.nii',
 'GIST-003_CT.nii',
 'GIST-004_CT.nii',
 'GIST-005_CT.nii',
 'GIST-006_CT.nii',
 'GIST-007_CT.nii',
 'GIST-008_CT.nii',
 'GIST-009_CT.nii',
 'GIST-010_CT.nii']

In [37]:
from med3pipe import split_validation

# Create imagesVal/labelsVal if not present
split_validation(paths, split_ratio=0.8, seed=2025, copy=True)

Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50


(196, 50)

In [38]:
from med3pipe import Train3DConfig, train_eval_densenet121_3d

res = train_eval_densenet121_3d(
    paths=paths,
    dataset_root=Path("gist"),               # resolver will use dataset_root/sheet.csv
    cfg=Train3DConfig(epochs=1, img_size=96, batch_size=2, device="cpu"),  # or "cpu"
)
print("Saved to:", res["out_dir"])

Saved to: c:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\vision3d_runs\densenet121_3d_20250916_155915


In [39]:
from pathlib import Path
from med3pipe import (
    find_default_sam3d_root, Sam3DPaths,
    Train3DConfig, train_eval_vit_3d
)

SAM3D_ROOT = find_default_sam3d_root()
paths = Sam3DPaths(sam3d_root=SAM3D_ROOT, category="gist", ct_name="ct_GIST")

res_ViT = train_eval_vit_3d(
    paths=paths,
    dataset_root=Path("gist"),
    dataset_name=None,
    img_size_3d=(96, 96, 96),       # keep aligned with cfg.img_size
    patch_size=(16, 16, 16),
    hidden_size=384, mlp_dim=1536,
    num_layers=8, num_heads=6,
    pos_embed="conv",
    cfg=Train3DConfig(
        epochs=1,
        img_size=96,
        batch_size=2,
        lr=2e-4,
        weight_decay=1e-4,
        device="cpu",               # or "cuda" to use GPU
    ),
)
print("Run dir:", res_ViT["out_dir"])
print("Val accuracy:", res_ViT["eval"]["acc"])
print("Val macro-F1:", res_ViT["eval"]["macro_f1"])

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

Run dir: c:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\vision3d_runs\vit3d_20250916_160254
Val accuracy: 0.5256410256410257
Val macro-F1: 0.3445378151260504


c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape